In [ ]:
# from google.colab import drive; drive.mount('/content/drive')  # uncomment for Colab
base_path = './'  # change to 'drive/MyDrive/NLP/' for Colab

In [ ]:
!pip install transformers torch scikit-learn matplotlib

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import re
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv(os.path.join(base_path, 'stackoverflow_posts.csv'))
if 'tag_id' not in df.columns:
    tag_map = {t: i for i, t in enumerate(sorted(df['tag_name'].dropna().unique()))}
    df['tag_id'] = df['tag_name'].map(tag_map)
df = df.dropna(subset=['title', 'tag_id'])

In [ ]:
def clean_text(text):
    return re.sub(r'[^\w\s]', '', text.lower())
df['title'] = df['title'].astype(str).apply(clean_text)
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
def tokenize(texts):
    return tokenizer(texts, truncation=True, padding=True, return_tensors='pt')

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['tag_id'], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.1, stratify=train_df['tag_id'], random_state=42)
class_counts = train_df['tag_id'].value_counts()
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.max()
class_weights = class_weights.sort_index()

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(df['tag_id'].unique()))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor(class_weights.values, dtype=torch.float).to(device))
class Dataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = tokenize([self.texts[idx]])
        return {k: v.squeeze() for k,v in enc.items()}, torch.tensor(self.labels[idx])
train_ds = Dataset(train_df['title'].tolist(), train_df['tag_id'].tolist())
val_ds = Dataset(val_df['title'].tolist(), val_df['tag_id'].tolist())
test_ds = Dataset(test_df['title'].tolist(), test_df['tag_id'].tolist())
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=16)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=16)
history = {'acc': [], 'f1': []}
epochs = 3
for epoch in range(epochs):
    model.train()
    for batch in train_loader:
        optimizer.zero_grad()
        inputs, labels = batch
        inputs = {k: v.to(device) for k,v in inputs.items()}
        labels = labels.to(device)
        outputs = model(**inputs)
        loss = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()
    model.eval()
    val_preds = []
    val_labels = []
    with torch.no_grad():
        for batch in val_loader:
            inputs, labels = batch
            inputs = {k: v.to(device) for k,v in inputs.items()}
            labels = labels.to(device)
            outputs = model(**inputs)
            preds = torch.argmax(outputs.logits, dim=1)
            val_preds.extend(preds.tolist())
            val_labels.extend(labels.tolist())
    acc = accuracy_score(val_labels, val_preds)
    f1 = f1_score(val_labels, val_preds, average='weighted')
    history['acc'].append(acc)
    history['f1'].append(f1)
    print(f'Epoch {epoch+1}: acc {acc}, f1 {f1}')

In [ ]:
model.eval()
test_preds = []
test_labels = []
with torch.no_grad():
    for batch in test_loader:
        inputs, labels = batch
        inputs = {k: v.to(device) for k,v in inputs.items()}
        labels = labels.to(device)
        outputs = model(**inputs)
        preds = torch.argmax(outputs.logits, dim=1)
        test_preds.extend(preds.tolist())
        test_labels.extend(labels.tolist())
acc = accuracy_score(test_labels, test_preds)
f1 = f1_score(test_labels, test_preds, average='weighted')
print(f'Test acc: {acc}, f1: {f1}')
plt.plot(history['acc'], label='acc')
plt.plot(history['f1'], label='f1')
plt.legend()
plt.show()